In [1]:
import os
import re
import urllib.request
from pathlib import Path

import faiss
import numpy as np
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

SMOKE = os.environ.get("COURSE_SMOKE") == "1"
torch.manual_seed(0)

## 1. Load and chunk the document

The handbook sits next to this notebook in `data/`. In Colab, the cell downloads it from the course repository.

Chunking splits the text at its headings, then cuts long sections into pieces of at most 120 words that overlap by 30 words.
The overlap keeps a sentence that straddles a cut in at least one chunk whole. Each chunk keeps its section heading, so it still makes sense on its own.

In [2]:
HANDBOOK = "week4-rag-handbook.md"
URL = "https://raw.githubusercontent.com/lemay-ai/CEG4195-SEG4180/main/data/week4-rag-handbook.md"
candidates = [Path("data") / HANDBOOK, Path("/code/data") / HANDBOOK]  # here, or the course container
path = next((p for p in candidates if p.exists()), None)
if path is None:
    path = Path(HANDBOOK)
    urllib.request.urlretrieve(URL, path)
text = path.read_text(encoding="utf-8")
print(len(text.split()), "words")

761 words


In [3]:
def chunk(text, size=120, overlap=30):
    """Split at '## ' headings, then into windows of `size` words that overlap by `overlap` words."""
    chunks = []
    for section in re.split(r"\n(?=## )", text):
        if not section.startswith("## "):
            continue  # the title and the note above the first section
        heading, _, body = section.partition("\n")
        heading = heading.lstrip("# ").strip()
        words = body.split()
        for start in range(0, max(len(words) - overlap, 1), size - overlap):
            piece = " ".join(words[start:start + size])
            if piece:
                chunks.append(f"{heading}: {piece}")
    return chunks


chunks = chunk(text)
print(len(chunks), "chunks")
for c in chunks[:3]:
    print("-", c[:120], "...")

10 chunks
- Opening hours: The makerspace is open from 8 am to 10 pm on weekdays and from 10 am to 6 pm on weekends. It is closed on ...
- Memberships and fees: A standard membership costs 65 dollars a month. Students with a valid student card pay 35 dollars  ...
- Guests: Each member may bring up to two guests at a time. Guests sign the waiver at the front desk before they touch any ...


## 2. Embed the chunks and build a FAISS index

The vectors have unit length, so the inner product is the cosine similarity. `IndexFlatIP` compares the question with every chunk.
That is exact, and fast enough for thousands of chunks.

In [4]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_vectors = embedder.encode(chunks, normalize_embeddings=True).astype(np.float32)

index = faiss.IndexFlatIP(chunk_vectors.shape[1])  # 384 dimensions
index.add(chunk_vectors)
print(index.ntotal, "vectors of", index.d, "dimensions")

10 vectors of 384 dimensions


In [5]:
def search(question, k=5):
    """The k chunks closest to the question: [(score, chunk number), ...]."""
    q = embedder.encode([question], normalize_embeddings=True).astype(np.float32)
    scores, ids = index.search(q, k)
    return list(zip(scores[0].tolist(), ids[0].tolist()))


question = "Can I bring a guest to use the laser cutter?"
hits = search(question)
for score, i in hits:
    print(f"{score:.3f}  [{i:2d}] {chunks[i][:100]}...")

0.664  [ 2] Guests: Each member may bring up to two guests at a time. Guests sign the waiver at the front desk b...
0.608  [ 4] Laser cutter: The laser cutter is booked in 30-minute slots, at most two slots per member per day. A...
0.321  [ 3] Safety training: Every new member takes the general safety orientation, a 90-minute session held eve...
0.274  [ 7] Wood shop: The wood shop has a table saw with a flesh-sensing brake, a bandsaw, a planer, a drill pr...
0.217  [ 0] Opening hours: The makerspace is open from 8 am to 10 pm on weekdays and from 10 am to 6 pm on weeke...


## 3. Rerank

The embedding model encodes the question and each chunk separately, then compares two vectors. That is fast, but coarse.
A **cross-encoder** reads the question and one chunk together, so it can check whether the chunk actually answers the question.
It is too slow to run on every chunk, so it only rescores the few candidates that the vector search returned.

In [6]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")


def rerank(question, hits, keep=2):
    """Rescore the candidates with the cross-encoder and keep the best `keep`."""
    scores = reranker.predict([(question, chunks[i]) for _, i in hits])
    order = np.argsort(-scores)
    return [(float(scores[j]), hits[j][1]) for j in order[:keep]]


for score, i in rerank(question, hits, keep=len(hits)):
    print(f"{score:7.2f}  [{i:2d}] {chunks[i][:100]}...")

   7.13  [ 2] Guests: Each member may bring up to two guests at a time. Guests sign the waiver at the front desk b...
  -1.59  [ 4] Laser cutter: The laser cutter is booked in 30-minute slots, at most two slots per member per day. A...
  -5.46  [ 3] Safety training: Every new member takes the general safety orientation, a 90-minute session held eve...
 -11.36  [ 0] Opening hours: The makerspace is open from 8 am to 10 pm on weekdays and from 10 am to 6 pm on weeke...
 -11.37  [ 7] Wood shop: The wood shop has a table saw with a flesh-sensing brake, a bandsaw, a planer, a drill pr...


Both put the guests chunk first here, but the reranker separates it far more clearly from the rest.
Its scores are not cosines: they are relevance scores with no fixed range. Section 4 has a question where the reranker changes the order.

## 4. Answer with a small local LLM

The prompt holds only the best chunks, numbered, and tells the model to answer from them and to say so when they do not hold the answer.

In [7]:
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct" if SMOKE else "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32)
llm.eval()


def generate(messages, max_new_tokens=80):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def answer(question, k=5, keep=2, show=True):
    """Retrieve k chunks, rerank, keep the best `keep`, and answer from them."""
    best = rerank(question, search(question, k), keep)
    context = "\n\n".join(f"[{n + 1}] {chunks[i]}" for n, (_, i) in enumerate(best))
    messages = [
        {"role": "system", "content": "Answer the question using only the context. "
                                      "If the context does not contain the answer, say you do not know."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    if show:
        print("Retrieved:", [i for _, i in best])
    return generate(messages)


print(answer(question))

Retrieved: [2, 4]


No, you cannot bring a guest to use the laser cutter.


The same question without retrieval: the model has never seen this makerspace, so it can only guess.

In [8]:
print(generate([{"role": "user", "content": "At the Rideau Makerspace, " + question[0].lower() + question[1:]}]))

Yes, at the Rideau Makerspace, you can certainly invite guests to use the laser cutter. The makerspace is designed for collaborative and creative projects, so it's an ideal place where people of all ages and skill levels can come together to work on their own or shared projects. If you're planning to bring someone along, make sure they have a valid membership or pass that allows them access


**Retrieval right, answer wrong.** Ask the same thing another way. This time the vector search puts the laser cutter chunk first, and the reranker moves the guests chunk back to the top.
The context now says guests may not use the laser cutter "even with a member present", yet a 1.5-billion-parameter model can still misread a rule like that.
Check retrieval and generation separately: a correct context does not guarantee a correct answer.

In [9]:
q = "Can my friend use the laser cutter if I am with them?"
print("Vector search:", [i for _, i in search(q)])
print("Reranked:     ", [i for _, i in rerank(q, search(q), keep=5)])
print(answer(q))

Vector search: [4, 2, 3, 7, 5]
Reranked:      [2, 4, 3, 7, 5]


Retrieved: [2, 4]


Yes.


More questions. The error code question shows a weakness of embeddings: "E37" is an exact string, and the embedding model has no special idea of what it means.
Check whether the right chunk was retrieved. Hybrid search adds a keyword index (BM25) for exactly this case.
The last question is not in the handbook at all: a good answer says so.

In [10]:
for q in ["How much does a student membership cost?",
          "My 3D printer shows E37. What should I do?",
          "What happens if the table saw brake fires twice?",
          "Does the makerspace have a welding area?"]:
    print("Q:", q)
    print("A:", answer(q), "\n")

Q: How much does a student membership cost?


Retrieved: [1, 8]


A: A student membership costs 35 dollars a month. 

Q: My 3D printer shows E37. What should I do?


Retrieved: [5, 6]


A: Load a new spool, then press Resume; do not cancel the print. 

Q: What happens if the table saw brake fires twice?


Retrieved: [7, 4]


A: The replacement is free the first time; after that, the member pays 120 dollars for the cartridge. 

Q: Does the makerspace have a welding area?


Retrieved: [0, 2]


A: I do not know. 



## 5. Scaling up: FAISS clusters the vectors

`IndexFlatIP` compares the question with every vector. With millions of chunks, an inverted-file index (`IndexIVFFlat`) first clusters the vectors with k-means,
then searches only the few clusters nearest the question (`nprobe`). This is the "Clustering with FAISS" slide.
Synthetic vectors stand in for a large collection: points scattered around 1,000 "topics", the way real embeddings bunch up by subject.
We compare the IVF index's answers with the exact search.

In [11]:
rng = np.random.default_rng(0)
n, d = (20_000 if SMOKE else 200_000), 384
topics = rng.standard_normal((1_000, d)).astype(np.float32)
big = topics[rng.integers(0, len(topics), n)] + 2.0 * rng.standard_normal((n, d)).astype(np.float32)
faiss.normalize_L2(big)
queries = topics[rng.integers(0, len(topics), 100)] + 2.0 * rng.standard_normal((100, d)).astype(np.float32)
faiss.normalize_L2(queries)

exact = faiss.IndexFlatIP(d)
exact.add(big)
_, truth = exact.search(queries, 10)

n_clusters = 256
ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, n_clusters, faiss.METRIC_INNER_PRODUCT)
ivf.train(big)  # k-means: 256 cluster centres
ivf.add(big)
for nprobe in [1, 4, 16, 64]:
    ivf.nprobe = nprobe
    _, found = ivf.search(queries, 10)
    recall = np.mean([len(set(f) & set(t)) / 10 for f, t in zip(found, truth)])
    print(f"nprobe={nprobe:3d}: searches {nprobe / n_clusters:6.1%} of the clusters, recall@10 = {recall:.2f}")

nprobe=  1: searches   0.4% of the clusters, recall@10 = 0.55
nprobe=  4: searches   1.6% of the clusters, recall@10 = 0.61
nprobe= 16: searches   6.2% of the clusters, recall@10 = 0.69


nprobe= 64: searches  25.0% of the clusters, recall@10 = 0.84


Searching more clusters finds more of the true neighbours, at the cost of more comparisons. Pure random vectors, with no topics, would cluster badly and need a much larger `nprobe`.

## Try it

- Change the chunk size to 30 and to 300 words. Which questions get worse, and why?
- Set `keep=1` and `keep=4` in `answer()`. Does more context help the small model, or confuse it?
- Replace the handbook with a document of your own.